# Phase 5: OCR & Document Intelligence
## Day 22: EasyOCRAndComparison

Date: 2026-04-24

### Learning objectives
- Understand what EasyOCR does.
- Learn EasyOCR setup and result format.
- Compare EasyOCR against Tesseract.
- Inspect confidence scores and bounding boxes.
- Choose the right OCR tool for a document task.

In [ ]:
import json
import re
import shutil
import textwrap
import time
from pprint import pprint

import numpy as np
import pandas as pd

try:
    from PIL import Image, ImageDraw, ImageFont, ImageFilter
    PIL_AVAILABLE = True
except Exception:
    PIL_AVAILABLE = False

try:
    import matplotlib.pyplot as plt
    MATPLOTLIB_AVAILABLE = True
except Exception:
    MATPLOTLIB_AVAILABLE = False

try:
    import pytesseract
    PYTESSERACT_AVAILABLE = True
except Exception:
    pytesseract = None
    PYTESSERACT_AVAILABLE = False

try:
    import easyocr
    EASYOCR_AVAILABLE = True
except Exception:
    easyocr = None
    EASYOCR_AVAILABLE = False

TESSERACT_BINARY_AVAILABLE = shutil.which("tesseract") is not None

def show(title, content):
    print("\n" + "=" * 76)
    print(title)
    print("=" * 76)
    print(textwrap.dedent(str(content)).strip())

print("Setup complete.")
print("PIL available:", PIL_AVAILABLE)
print("pytesseract available:", PYTESSERACT_AVAILABLE)
print("Tesseract binary available:", TESSERACT_BINARY_AVAILABLE)
print("EasyOCR available:", EASYOCR_AVAILABLE)

In [ ]:
sample_receipt_text = '''
BERLIN BAKERY
Date: 2026-04-24
Latte             4.20 EUR
Croissant         3.10 EUR
Bagel             5.50 EUR
Total            12.80 EUR
'''

sample_german_text = '''
GUTEN MORGEN
Ich möchte einen Kaffee.
Die Rechnung ist 12.80 EUR.
'''

sample_noisy_text = '''
Invoice ID: INV-2048
Campaign: Yoga Studio Trial
Spend: 650 EUR
Clicks: 2100
Conversions: 165
'''

print(sample_receipt_text)

## 1. Create sample images

We will create clean and noisy images inline.

This makes the notebook runnable without external image files.

In [ ]:
def create_text_image(text, width=820, height=360, font_size=24, noisy=False):
    if not PIL_AVAILABLE:
        return None

    image = Image.new("RGB", (width, height), color="white")
    draw = ImageDraw.Draw(image)

    try:
        font = ImageFont.truetype("DejaVuSansMono.ttf", font_size)
    except Exception:
        font = ImageFont.load_default()

    draw.multiline_text((30, 30), text.strip(), fill="black", font=font, spacing=10)

    if noisy:
        arr = np.array(image).astype(np.int16)
        noise = np.random.normal(loc=0, scale=18, size=arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        image = Image.fromarray(arr)
        image = image.filter(ImageFilter.GaussianBlur(radius=0.4))

    return image

receipt_image = create_text_image(sample_receipt_text)
german_image = create_text_image(sample_german_text)
noisy_image = create_text_image(sample_noisy_text, noisy=True)

if receipt_image is not None:
    display(receipt_image)
else:
    print("PIL is not available.")

In [ ]:
if noisy_image is not None:
    display(noisy_image)
else:
    print("PIL is not available.")

## 2. EasyOCR setup

EasyOCR is a deep learning based OCR library.

It is often strong on natural images, multilingual text, and text that is not perfectly aligned. It can be slower than Tesseract because it loads neural models.

In [ ]:
easyocr_setup_notes = '''
Install:
pip install easyocr

Basic usage:
import easyocr
reader = easyocr.Reader(["en"])
results = reader.readtext(image_array)

Language examples:
reader = easyocr.Reader(["en"])
reader = easyocr.Reader(["de", "en"])
reader = easyocr.Reader(["tr", "en"])

Result format:
[
  (bounding_box, text, confidence),
  ...
]
'''

print(easyocr_setup_notes)

In [ ]:
def mock_easyocr_results(kind="receipt"):
    if kind == "german":
        return [
            ([[30, 30], [210, 30], [210, 55], [30, 55]], "GUTEN MORGEN", 0.97),
            ([[30, 74], [390, 74], [390, 100], [30, 100]], "Ich möchte einen Kaffee.", 0.93),
            ([[30, 118], [420, 118], [420, 145], [30, 145]], "Die Rechnung ist 12.80 EUR.", 0.91),
        ]
    if kind == "noisy":
        return [
            ([[30, 30], [300, 30], [300, 55], [30, 55]], "Invoice ID: INV-2048", 0.88),
            ([[30, 74], [430, 74], [430, 100], [30, 100]], "Campaign: Yoga Studio Trial", 0.86),
            ([[30, 118], [260, 118], [260, 145], [30, 145]], "Spend: 650 EUR", 0.84),
            ([[30, 162], [250, 162], [250, 188], [30, 188]], "Clicks: 2100", 0.82),
            ([[30, 206], [330, 206], [330, 232], [30, 232]], "Conversions: 165", 0.83),
        ]
    return [
        ([[30, 30], [250, 30], [250, 55], [30, 55]], "BERLIN BAKERY", 0.98),
        ([[30, 74], [300, 74], [300, 100], [30, 100]], "Date: 2026-04-24", 0.96),
        ([[30, 118], [430, 118], [430, 145], [30, 145]], "Latte 4.20 EUR", 0.94),
        ([[30, 162], [460, 162], [460, 188], [30, 188]], "Croissant 3.10 EUR", 0.93),
        ([[30, 206], [430, 206], [430, 232], [30, 232]], "Bagel 5.50 EUR", 0.95),
        ([[30, 250], [430, 250], [430, 276], [30, 276]], "Total 12.80 EUR", 0.96),
    ]

def run_easyocr(image, languages=None, kind="receipt"):
    languages = languages or ["en"]

    if image is not None and EASYOCR_AVAILABLE:
        try:
            reader = easyocr.Reader(languages, gpu=False)
            image_array = np.array(image)
            return reader.readtext(image_array)
        except Exception as error:
            print("Real EasyOCR failed. Using mock EasyOCR results.")
            print("Error:", error)
            return mock_easyocr_results(kind)

    print("EasyOCR is not available. Using mock EasyOCR results.")
    return mock_easyocr_results(kind)

easy_results = run_easyocr(receipt_image, languages=["en"], kind="receipt")
pprint(easy_results[:2])

## 3. EasyOCR result format

Each EasyOCR result has three parts.

The parts are bounding box, detected text, and confidence score.

In [ ]:
def easyocr_to_dataframe(results):
    rows = []
    for box, text, confidence in results:
        xs = [point[0] for point in box]
        ys = [point[1] for point in box]
        rows.append({
            "text": text,
            "confidence": float(confidence),
            "left": min(xs),
            "top": min(ys),
            "right": max(xs),
            "bottom": max(ys),
            "width": max(xs) - min(xs),
            "height": max(ys) - min(ys),
        })
    return pd.DataFrame(rows)

easy_df = easyocr_to_dataframe(easy_results)
easy_df

In [ ]:
def join_easyocr_text(results):
    return "\n".join(text for _, text, _ in results)

easy_text = join_easyocr_text(easy_results)
show("EasyOCR text", easy_text)

## 4. Tesseract baseline

Tesseract is a classic OCR engine.

It is often fast and strong on clean scanned documents, receipts, invoices, and simple layouts.

In [ ]:
def mock_tesseract_text(kind="receipt"):
    if kind == "german":
        return sample_german_text.strip()
    if kind == "noisy":
        return sample_noisy_text.strip()
    return sample_receipt_text.strip()

def run_tesseract(image, lang="eng", config="--psm 6 --oem 3", kind="receipt"):
    if image is not None and PYTESSERACT_AVAILABLE and TESSERACT_BINARY_AVAILABLE:
        try:
            return pytesseract.image_to_string(image, lang=lang, config=config)
        except Exception as error:
            print("Real Tesseract failed. Using mock Tesseract text.")
            print("Error:", error)
            return mock_tesseract_text(kind)

    print("pytesseract or Tesseract is not available. Using mock Tesseract text.")
    return mock_tesseract_text(kind)

tess_text = run_tesseract(receipt_image, lang="eng", kind="receipt")
show("Tesseract text", tess_text)

In [ ]:
def mock_tesseract_data(kind="receipt"):
    if kind == "noisy":
        rows = [
            {"text": "Invoice", "conf": 82, "left": 30, "top": 30, "width": 90, "height": 24},
            {"text": "ID:", "conf": 80, "left": 130, "top": 30, "width": 40, "height": 24},
            {"text": "INV-2048", "conf": 78, "left": 180, "top": 30, "width": 120, "height": 24},
            {"text": "Campaign:", "conf": 76, "left": 30, "top": 74, "width": 130, "height": 24},
            {"text": "Yoga", "conf": 75, "left": 170, "top": 74, "width": 70, "height": 24},
        ]
    else:
        rows = [
            {"text": "BERLIN", "conf": 95, "left": 30, "top": 30, "width": 90, "height": 24},
            {"text": "BAKERY", "conf": 96, "left": 130, "top": 30, "width": 100, "height": 24},
            {"text": "Date:", "conf": 94, "left": 30, "top": 74, "width": 70, "height": 24},
            {"text": "2026-04-24", "conf": 93, "left": 110, "top": 74, "width": 160, "height": 24},
            {"text": "Latte", "conf": 92, "left": 30, "top": 118, "width": 80, "height": 24},
            {"text": "4.20", "conf": 90, "left": 310, "top": 118, "width": 60, "height": 24},
            {"text": "Total", "conf": 94, "left": 30, "top": 250, "width": 80, "height": 24},
            {"text": "12.80", "conf": 91, "left": 300, "top": 250, "width": 75, "height": 24},
        ]
    return pd.DataFrame(rows)

def get_tesseract_data(image, lang="eng", config="--psm 6 --oem 3", kind="receipt"):
    if image is not None and PYTESSERACT_AVAILABLE and TESSERACT_BINARY_AVAILABLE:
        try:
            data = pytesseract.image_to_data(
                image,
                lang=lang,
                config=config,
                output_type=pytesseract.Output.DATAFRAME
            )
            data = data.dropna(subset=["text"])
            data = data[data["text"].astype(str).str.strip() != ""]
            return data[["text", "conf", "left", "top", "width", "height"]].reset_index(drop=True)
        except Exception as error:
            print("Real Tesseract data failed. Using mock data.")
            print("Error:", error)

    print("Using mock Tesseract bounding boxes.")
    return mock_tesseract_data(kind)

tess_df = get_tesseract_data(receipt_image)
tess_df

## 5. Compare outputs

A fair comparison should look at text quality, confidence, speed, setup, language support, and layout needs.

Here we create a small comparison table.

In [ ]:
def normalize_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9.]+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def token_set(text):
    return set(normalize_text(text).split())

def token_recall(reference, prediction):
    ref = token_set(reference)
    pred = token_set(prediction)
    if not ref:
        return 0.0
    return len(ref & pred) / len(ref)

reference_text = sample_receipt_text
easy_recall = token_recall(reference_text, easy_text)
tess_recall = token_recall(reference_text, tess_text)

comparison = pd.DataFrame([
    {
        "engine": "EasyOCR",
        "token_recall": round(easy_recall, 3),
        "avg_confidence": round(easy_df["confidence"].mean(), 3),
        "result_style": "line level with boxes"
    },
    {
        "engine": "Tesseract",
        "token_recall": round(tess_recall, 3),
        "avg_confidence": round(tess_df["conf"].mean() / 100, 3),
        "result_style": "word level with boxes"
    },
])

comparison

In [ ]:
tool_comparison = pd.DataFrame([
    {
        "feature": "Best on clean scans",
        "Tesseract": "Strong",
        "EasyOCR": "Good"
    },
    {
        "feature": "Natural photos",
        "Tesseract": "Can struggle",
        "EasyOCR": "Often stronger"
    },
    {
        "feature": "Speed after setup",
        "Tesseract": "Usually faster",
        "EasyOCR": "Can be slower"
    },
    {
        "feature": "Deep learning model download",
        "Tesseract": "No",
        "EasyOCR": "Yes"
    },
    {
        "feature": "Simple Python setup",
        "Tesseract": "Needs system binary",
        "EasyOCR": "pip package, but heavier"
    },
    {
        "feature": "Bounding boxes",
        "Tesseract": "Word level",
        "EasyOCR": "Line or text region level"
    },
])

tool_comparison

## 6. Compare on noisy text

OCR tools behave differently when the image is noisy.

In real work, test both engines on your real document type before choosing one.

In [ ]:
noisy_easy_results = run_easyocr(noisy_image, languages=["en"], kind="noisy")
noisy_easy_text = join_easyocr_text(noisy_easy_results)
noisy_easy_df = easyocr_to_dataframe(noisy_easy_results)

noisy_tess_text = run_tesseract(noisy_image, lang="eng", kind="noisy")
noisy_tess_df = get_tesseract_data(noisy_image, kind="noisy")

show("Noisy EasyOCR text", noisy_easy_text)
show("Noisy Tesseract text", noisy_tess_text)

In [ ]:
noisy_reference = sample_noisy_text

noisy_comparison = pd.DataFrame([
    {
        "engine": "EasyOCR",
        "token_recall": round(token_recall(noisy_reference, noisy_easy_text), 3),
        "avg_confidence": round(noisy_easy_df["confidence"].mean(), 3)
    },
    {
        "engine": "Tesseract",
        "token_recall": round(token_recall(noisy_reference, noisy_tess_text), 3),
        "avg_confidence": round(noisy_tess_df["conf"].mean() / 100, 3)
    },
])

noisy_comparison

## 7. Draw bounding boxes

Bounding boxes help you debug OCR.

They also help later when you need layout-aware extraction.

In [ ]:
def draw_easyocr_boxes(image, results):
    if image is None or not PIL_AVAILABLE:
        print("Image drawing is not available.")
        return None

    boxed = image.copy()
    draw = ImageDraw.Draw(boxed)

    for box, text, confidence in results:
        points = [(int(x), int(y)) for x, y in box]
        draw.line(points + [points[0]], fill="red", width=2)
        draw.text(points[0], f"{confidence:.2f}", fill="red")

    return boxed

easy_boxed = draw_easyocr_boxes(receipt_image, easy_results)

if easy_boxed is not None:
    display(easy_boxed)

In [ ]:
def draw_tesseract_boxes(image, data):
    if image is None or not PIL_AVAILABLE:
        print("Image drawing is not available.")
        return None

    boxed = image.copy()
    draw = ImageDraw.Draw(boxed)

    for _, row in data.iterrows():
        left = int(row["left"])
        top = int(row["top"])
        right = left + int(row["width"])
        bottom = top + int(row["height"])
        draw.rectangle([left, top, right, bottom], outline="blue", width=2)

    return boxed

tess_boxed = draw_tesseract_boxes(receipt_image, tess_df)

if tess_boxed is not None:
    display(tess_boxed)

## 8. Choose an OCR engine

There is no universal winner.

Choose based on your image type, language, speed needs, and deployment environment.

In [ ]:
def recommend_ocr_engine(clean_scan=True, natural_photo=False, needs_fast=True, system_install_ok=True):
    score = {"Tesseract": 0, "EasyOCR": 0}

    if clean_scan:
        score["Tesseract"] += 2
        score["EasyOCR"] += 1

    if natural_photo:
        score["EasyOCR"] += 2

    if needs_fast:
        score["Tesseract"] += 1

    if not system_install_ok:
        score["EasyOCR"] += 1
    else:
        score["Tesseract"] += 1

    winner = max(score, key=score.get)
    return {
        "recommendation": winner,
        "score": score
    }

scenarios = [
    {"clean_scan": True, "natural_photo": False, "needs_fast": True, "system_install_ok": True},
    {"clean_scan": False, "natural_photo": True, "needs_fast": False, "system_install_ok": False},
    {"clean_scan": True, "natural_photo": True, "needs_fast": False, "system_install_ok": True},
]

for scenario in scenarios:
    print("\nScenario:")
    pprint(scenario)
    pprint(recommend_ocr_engine(**scenario))

## 9. Field extraction after OCR

The OCR engine gives text.

The next step is usually field extraction, validation, and quality checks.

In [ ]:
def extract_receipt_total(text):
    clean = re.sub(r"\s+", " ", text)
    match = re.search(r"Total\s+([0-9]+\.[0-9]{2})\s+EUR", clean, flags=re.IGNORECASE)
    return float(match.group(1)) if match else None

def extract_invoice_fields(text):
    clean = re.sub(r"\s+", " ", text)
    return {
        "invoice_id": find.group(1) if (find := re.search(r"Invoice ID:\s*([A-Z]+-[0-9]+)", clean)) else None,
        "campaign": find.group(1) if (find := re.search(r"Campaign:\s*([A-Za-z\s]+?)\s+Spend:", clean)) else None,
        "spend_eur": int(find.group(1)) if (find := re.search(r"Spend:\s*(\d+)\s+EUR", clean)) else None,
        "clicks": int(find.group(1)) if (find := re.search(r"Clicks:\s*(\d+)", clean)) else None,
        "conversions": int(find.group(1)) if (find := re.search(r"Conversions:\s*(\d+)", clean)) else None,
    }

print("Receipt total from EasyOCR:", extract_receipt_total(easy_text))
print("Receipt total from Tesseract:", extract_receipt_total(tess_text))

print("\nInvoice fields from noisy EasyOCR:")
pprint(extract_invoice_fields(noisy_easy_text))

## Tricky bits

OCR comparison can be misleading if you only test one image.

Use several real examples, check field-level accuracy, and include bad images in your test set.

In [ ]:
tricky_cases = pd.DataFrame([
    {
        "mistake": "Only comparing full text",
        "why_it_hurts": "A small OCR error can break an important field",
        "better_check": "Measure field-level accuracy"
    },
    {
        "mistake": "Ignoring speed",
        "why_it_hurts": "A slow OCR engine can block batch processing",
        "better_check": "Measure runtime on many files"
    },
    {
        "mistake": "Ignoring setup",
        "why_it_hurts": "Deployment may fail if system packages are missing",
        "better_check": "Test inside the target environment"
    },
    {
        "mistake": "Trusting confidence too much",
        "why_it_hurts": "Confidence is useful but not perfect",
        "better_check": "Validate important fields with rules"
    },
])

tricky_cases

In [ ]:
def validate_receipt_total(total):
    if total is None:
        return "Missing total"
    if total <= 0:
        return "Total must be positive"
    if total > 1000:
        return "Total looks too high for this receipt"
    return "Looks valid"

test_totals = [12.80, None, -5, 5000]

for total in test_totals:
    print(total, "=>", validate_receipt_total(total))

## Trick questions

1. Is EasyOCR always better than Tesseract?

<details>
<summary>Answer</summary>

No. EasyOCR can be better for natural images, but Tesseract can be faster and very strong on clean scans.

</details>

2. What is the EasyOCR result format?

<details>
<summary>Answer</summary>

It returns bounding box, text, and confidence for each detected text region.

</details>

3. Why compare field-level accuracy?

<details>
<summary>Answer</summary>

Because a full text score can look okay even when an important field like total, invoice ID, or date is wrong.

</details>

4. Why can EasyOCR be slower?

<details>
<summary>Answer</summary>

It loads and runs deep learning models.

</details>

5. When might Tesseract be a better choice?

<details>
<summary>Answer</summary>

For clean scans, simple layouts, fast processing, and environments where Tesseract is already installed.

</details>

## Exercises

Fill in each `___`. Run the cell to check your answer.

In [ ]:
# Exercise 1
# Create an EasyOCR language list for English and German.

languages = ___

assert languages == ["en", "de"] or languages == ["de", "en"]
print("Exercise 1 passed.")

In [ ]:
# Exercise 2
# Run EasyOCR using the helper function.

results = ___

assert isinstance(results, list)
assert len(results) > 0
assert len(results[0]) == 3
print("Exercise 2 passed.")

In [ ]:
# Exercise 3
# Convert EasyOCR results to a DataFrame.

df_easy = ___

assert isinstance(df_easy, pd.DataFrame)
assert {"text", "confidence", "left", "top", "right", "bottom"}.issubset(df_easy.columns)
print("Exercise 3 passed.")

In [ ]:
# Exercise 4
# Join EasyOCR text into one string.

joined_text = ___

assert isinstance(joined_text, str)
assert "Total" in joined_text
print("Exercise 4 passed.")

In [ ]:
# Exercise 5
# Run Tesseract using the helper function.

text_tess = ___

assert isinstance(text_tess, str)
assert "Total" in text_tess
print("Exercise 5 passed.")

In [ ]:
# Exercise 6
# Calculate token recall for EasyOCR output.

recall = ___

assert 0 <= recall <= 1
print("Exercise 6 passed.")

In [ ]:
# Exercise 7
# Recommend an OCR engine for a clean scan where speed matters.

recommendation = ___

assert recommendation["recommendation"] in ["Tesseract", "EasyOCR"]
assert recommendation["recommendation"] == "Tesseract"
print("Exercise 7 passed.")

## Solutions

<details>
<summary>Exercise 1 solution</summary>

```python
languages = ["en", "de"]
```

</details>

<details>
<summary>Exercise 2 solution</summary>

```python
results = run_easyocr(receipt_image, languages=["en"], kind="receipt")
```

</details>

<details>
<summary>Exercise 3 solution</summary>

```python
df_easy = easyocr_to_dataframe(results)
```

</details>

<details>
<summary>Exercise 4 solution</summary>

```python
joined_text = join_easyocr_text(results)
```

</details>

<details>
<summary>Exercise 5 solution</summary>

```python
text_tess = run_tesseract(receipt_image, lang="eng", kind="receipt")
```

</details>

<details>
<summary>Exercise 6 solution</summary>

```python
recall = token_recall(sample_receipt_text, joined_text)
```

</details>

<details>
<summary>Exercise 7 solution</summary>

```python
recommendation = recommend_ocr_engine(
    clean_scan=True,
    natural_photo=False,
    needs_fast=True,
    system_install_ok=True
)
```

</details>

## Cumulative review exercises

These mix topics from Days 12 to 21. Fill in `___` and run each cell.

In [ ]:
# Review 1: Transformers
# Complete the sentence.

attention_word = ___

assert attention_word == "tokens"
print("Review 1 passed.")

In [ ]:
# Review 2: Hugging Face
# Fill the quick helper for pretrained tasks.

hf_helper = ___

assert hf_helper == "pipeline"
print("Review 2 passed.")

In [ ]:
# Review 3: Fine-tuning BERT
# Pick the common class for training.

trainer_class = ___

assert trainer_class == "Trainer"
print("Review 3 passed.")

In [ ]:
# Review 4: Complaint classification
# Create a label mapping.

label_to_id = ___

assert isinstance(label_to_id, dict)
assert len(label_to_id) >= 3
assert all(isinstance(v, int) for v in label_to_id.values())
print("Review 4 passed.")

In [ ]:
# Review 5: OpenAI API
# Fill the standard chat roles.

roles = ___

assert roles == ["system", "user", "assistant"]
print("Review 5 passed.")

In [ ]:
# Review 6: Ollama
# Fill the default local generate endpoint.

ollama_url = ___

assert ollama_url == "http://localhost:11434/api/generate"
print("Review 6 passed.")

In [ ]:
# Review 7: Prompt engineering
# Choose the prompting style that uses no examples.

prompt_style = ___

assert prompt_style.lower() == "zero-shot"
print("Review 7 passed.")

In [ ]:
# Review 8: Structured output
# Parse JSON text.

json_text = '{"campaign": "Demo", "clicks": 100}'
parsed = ___

assert parsed["clicks"] == 100
print("Review 8 passed.")

In [ ]:
# Review 9: Information extraction
# Calculate conversion rate.

record = {"clicks": 2000, "conversions": 120}
conversion_rate = ___

assert abs(conversion_rate - 0.06) < 1e-9
print("Review 9 passed.")

In [ ]:
# Review 10: Tesseract basics
# Choose the Tesseract language code for German.

german_lang_code = ___

assert german_lang_code == "deu"
print("Review 10 passed.")

## Cumulative review solutions

<details>
<summary>Show solutions</summary>

```python
# Review 1
attention_word = "tokens"

# Review 2
hf_helper = "pipeline"

# Review 3
trainer_class = "Trainer"

# Review 4
label_to_id = {"billing": 0, "delivery": 1, "technical": 2}

# Review 5
roles = ["system", "user", "assistant"]

# Review 6
ollama_url = "http://localhost:11434/api/generate"

# Review 7
prompt_style = "zero-shot"

# Review 8
parsed = json.loads(json_text)

# Review 9
conversion_rate = record["conversions"] / record["clicks"]

# Review 10
german_lang_code = "deu"
```

</details>

In [ ]:
cheat_sheet = '''
DAY 22 CHEAT SHEET: EASYOCR AND COMPARISON

EasyOCR:
- pip install easyocr
- reader = easyocr.Reader(["en"])
- results = reader.readtext(image_array)
- result format: bounding_box, text, confidence
- often strong on natural images and multilingual text

Tesseract:
- needs system binary plus pytesseract
- often fast and strong on clean scans
- language codes: eng, deu, tur
- common config: --psm 6 --oem 3
- can return word-level boxes with image_to_data

Comparison tips:
- Test on real documents.
- Compare field-level accuracy.
- Check speed and deployment setup.
- Use confidence scores carefully.
- Validate extracted fields after OCR.

Rule of thumb:
- Clean scanned documents: try Tesseract first.
- Natural photos or difficult text: try EasyOCR too.
- Production systems may use both and choose the better result.
'''

print(cheat_sheet)

## Next up: Day 23 — OpenCVPreprocessing

You will learn grayscale, thresholding, deskewing, and denoising to improve OCR quality.